# Custom RAG Chatbot Project

TODO: In this cell, write an explanation of which dataset you have chosen and why it is appropriate for this task

`./data/2023_fashion_trends.csv` was chosen mostly for its simplicity in data structure and question answer format. Simplest thing that could possibly work!

In [12]:
openapi_key = "<secret>"

## Data Wrangling

TODO: In the cells below, load your chosen dataset into a `pandas` dataframe with a column named `"text"`. This column should contain all of your text data, separated into at least 20 rows.

In [1]:
!find

.
./.ipynb_checkpoints
./.ipynb_checkpoints/project-checkpoint.ipynb
./project.ipynb
./data
./data/.ipynb_checkpoints
./data/character_descriptions.csv
./data/nyc_food_scrap_drop_off_sites.csv
./data/2023_fashion_trends.csv


In [2]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

In [3]:
data = pd.read_csv('./data/2023_fashion_trends.csv')
data['text'] = data['Trends'] + "\n" + data['Source']
data

,URL,Trends,Source,text
0,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Red. Glossy red hues took ...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend: Red. Glossy red hues took ...
1,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Cargo Pants. Utilitarian w...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend: Cargo Pants. Utilitarian w...
2,https://www.refinery29.com/en-us/fashion-trend...,"2023 Fashion Trend: Sheer Clothing. ""Bare it a...",7 Fashion Trends That Will Take Over 2023 — Sh...,"2023 Fashion Trend: Sheer Clothing. ""Bare it a..."
3,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Denim Reimagined. From dou...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend: Denim Reimagined. From dou...
4,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Shine For The Daytime. The...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend: Shine For The Daytime. The...
...,...,...,...,...
77,https://www.whowhatwear.com/spring-summer-2023...,"If lime green isn't your vibe, rest assured th...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...,"If lime green isn't your vibe, rest assured th..."
78,https://www.whowhatwear.com/spring-summer-2023...,"""As someone who can clearly (not fondly) remem...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...,"""As someone who can clearly (not fondly) remem..."
79,https://www.whowhatwear.com/spring-summer-2023...,"""Combine this design shift with the fact that ...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...,"""Combine this design shift with the fact that ..."
80,https://www.whowhatwear.com/spring-summer-2023...,Thought party season ended at the stroke of mi...,Spring/Summer 2023 Fashion Trends: 21 Expert-A...,Thought party season ended at the stroke of mi...


## Custom Query Completion

TODO: In the cells below, compose a custom query using your chosen dataset and retrieve results from an OpenAI `Completion` model. You may copy and paste any useful code from the course materials.

BUG: All the `Completion` models such as `text-davinci-003` are retired
BUG: `pip openai v0.26` doesn't support `ChatCompletion` models such as `gpt4o`
BUG: `pip install openai --upgrade` fails inside workspace

```
response = openai.Completion.create(
    model="text-davinci-003",  
    prompt="Hello world!",
    max_tokens=150,
    temperature=0.7,
    n=1
)
# InvalidRequestError: The model `text-davinci-003` has been deprecated, learn more here: https://platform.openai.com/docs/deprecations



completion = openai.ChatCompletion.create(
  model="gpt-4.1",
  messages=[
    {"role": "developer", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"}
  ]
)
# AttributeError: module 'openai' has no attribute 'ChatCompletion'



curl https://api.openai.com/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer sk-proj-***" \
  -d '{
    "model": "gpt-4.1",
    "messages": [
      {
        "role": "developer",
        "content": "You are a helpful assistant."
      },
      {
        "role": "user",
        "content": "Hello!"
      }
    ]
  }'



In [4]:
# !pip install --upgrade openai --quiet
!pip show openai

Name: openai
Version: 0.26.1
Summary: Python client library for the OpenAI API
Home-page: https://github.com/openai/openai-python
Author: OpenAI
Author-email: support@openai.com
License: 
Location: /opt/venv/lib/python3.9/site-packages
Requires: aiohttp, requests, tqdm
Required-by: 


In [5]:
import requests; 

def query(messages):
    response = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {openapi_key}"
        },
        json={
            "model": "gpt-4.1",
            "messages": messages
        }
    )
    return response.json()

query([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"}
])

{'id': 'chatcmpl-BwFYKxyb9AIdXZoajU6o1TjwV3D2w',
 'object': 'chat.completion',
 'created': 1753222508,
 'model': 'gpt-4.1-2025-04-14',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Hello! How can I help you today?',
    'refusal': None,
    'annotations': []},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 19,
  'completion_tokens': 9,
  'total_tokens': 28,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 0,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': 'fp_6608a0a96a'}

In [6]:
context = "\n\n".join(data['text'].tolist())
query([
    {
        "role": "system", 
        "content": f"""
        You are a fashion advisor suggesting stylish outfits for user events. 
        Use the following CONTEXT to guide you: 
        {context}
        """
    },
    {
        "role": "user", 
        "content": "autumn evening event"
    },
])

{'id': 'chatcmpl-BwFYLucOymePxPsVtMJnKrrxePnvx',
 'object': 'chat.completion',
 'created': 1753222509,
 'model': 'gpt-4.1-2025-04-14',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Absolutely, let’s craft a stylish look for an autumn evening event using 2023’s hottest trends!\n\n**1. The Statement Color:**  \nLean into the red trend for autumn—juicy, glossy reds with orange undertones are perfect for seasonal warmth and sophistication.\n\n**2. Outfit Idea (Chic, On-Trend & Elegant):**\n\n**Option 1: Modern Power Suit with Autumn Flair**\n- **Red tailored pantsuit:** Opt for a well-cut suit in a vibrant, glossy red (Tory Burch/Ferragamo vibes). The color pops for evening and the tailoring keeps it refined.\n- **Sheer blouse or mesh top:** Layer a delicately sheer, long-sleeve blouse in a matching or tonal shade under the blazer, embracing the “sheer clothing” trend in a subtle, sophisticated way.\n- **Accessories:**  \n   - **Sculptural metallic statement

In [7]:
def rag_query(prompt, context):   
    response = query([
        {
            "role": "system", 
            "content": f"""
            You are a fashion advisor suggesting stylish outfits for user events. 
            Use the following CONTEXT to guide you: 
            {context}
            """
        },
        {
            "role": "user", 
            "content": prompt # "autumn evening event:\n"
        },
    ])
    return response['choices'][0]['message']['content']

## Custom Performance Demonstration

TODO: In the cells below, demonstrate the performance of your custom query using at least 2 questions. For each question, show the answer from a basic `Completion` model query as well as the answer from your custom query.

### Question 1

In [8]:
context = "<empty>"
advise  = rag_query("autumn evening event", context)
display(Markdown(advise))

For an autumn evening event, aim for a chic look that keeps you warm and sophisticated:

**Women:**  
- A long-sleeved, midi or maxi wrap dress in a rich autumn hue like burgundy, forest green, or deep navy.
- Add a tailored wool-blend coat or a faux fur stole for extra warmth.
- Pair with heeled ankle boots or elegant pointed-toe pumps.
- Accessorize with statement earrings or a structured clutch.
- Consider opaque tights for extra warmth.

**Men:**  
- A crisp button-down shirt layered under a lightweight merino wool sweater or a tailored blazer.
- Choose tailored trousers in a dark neutral shade (charcoal, navy, or deep brown).
- Add a smart wool overcoat or a trench for the outdoors.
- Leather dress shoes or chelsea boots complete the look.
- Accessorize with a sleek watch or a silk pocket square.

Both looks are timeless and adaptable depending on the formality of your event! Let me know if you need ideas for a specific dress code.

In [9]:
context = "\n\n".join(data['text'].tolist())
advise  = rag_query("autumn evening event", context)
display(Markdown(advise))

Absolutely! For an autumn evening event, you have a wonderful opportunity to embrace both cozy sophistication and trending glamour. Here’s a fashion-forward outfit idea inspired by 2023’s top trends:

---

**Outfit Concept: Autumn Evening Chic**

### 1. **Statement Color & Fabric**
- **Dress:** Opt for a long-sleeved or midi-length slip dress in a **glossy red** (the ‘it’ color of 2023), or a rich jewel tone like cobalt blue or deep green. Think of luxe fabrics: satin, silk, or velvet for an autumnal feel and beautiful drape.
    - *Alternative:* Sheer sleeves or a touch of sheer paneling taps into the sheer dressing trend without overexposing.
    - *More daring?* Try a metallic slip dress in liquid silver or gold for a dazzling yet chic look.

### 2. **Layering/Warmth**
- **Outerwear:** Add a sharply **tailored cinched blazer** or a cropped leather moto jacket (channeling the moto-inspired trend) for warmth and structure.
- *Extra cozy?* Try a luxe faux fur stole or fringed shawl for texture and cold protection.

### 3. **Accessories**
- **Bag:** Choose a **sculptural bag** in a neutral (camel, ivory) or bold red for a pop.
- **Jewelry:** Add **sculptural statement earrings** (shoulder-grazing drops are huge this season) for drama. If your look is more minimal, go for romantic 3D floral or rosette details on a bracelet or ring.

### 4. **Shoes**
- Slip into pointed-toe **kitten-heel pumps** or strappy heels in white, metallic, or a juicy red for a trend-forward finish.
- Prefer closed shoes as it gets chilly: metallic or patent leather mules/ankle boots work well.

### 5. **Extra Touches**
- If you love the idea of texture, indulge in accessories or a skirt with refined **fringe** or embellishment.
- For a playful nod to balletcore, consider a tulle layer peeking beneath a skirt, or pair with elegant satin ballet flats.

---

**Example Look:**  
- Glossy red satin midi dress with a low back or subtle sheer sleeves  
- Cropped black leather moto jacket draped over your shoulders  
- Sculptural ivory clutch  
- Silver or sculptural drop earrings  
- Red pointed-toe kitten heels  
- Deep wine lipstick and softly waved hair

**Or for a more minimal look:**  
- Cobalt blue tailored pants, cream knit top, and an elegantly oversized camel blazer  
- Silver sculptural earrings  
- White or metallic heeled mules  
- Chunky chain link bag

---

**Final tip:** Play with layers and texture to keep warm, and let your accessories do the talking. Autumn evenings are for rich color, exciting accessories, and just a hint of sparkle or sheen.

Would you like suggestions tailored toward a specific type of venue/event or your personal style?

### Question 2

In [10]:
context = "<empty>"
advise  = rag_query("summer gala ball", context)
display(Markdown(advise))

A summer gala ball calls for elegance with a fresh, lighthearted twist. Here are some stylish outfit ideas:

**For Women:**
- **Gown:** Opt for a floor-length gown in breathable fabrics such as chiffon, silk, or lightweight satin. Colors like pastel blues, blush pink, lavender, emerald green, or classic navy are perfect for summer.
- **Details:** Look for details like an off-the-shoulder neckline, delicate beading, or flowing ruffles for a romantic touch.
- **Accessories:** Pair with strappy metallic heels, a clutch bag, and sparkling drop earrings. Consider an updo or soft waves for a polished look.
- **Makeup:** Keep it fresh and luminous with a soft highlight and a pop of color on the lips.

**For Men:**
- **Suit/Tuxedo:** Choose a lightweight tuxedo or tailored suit in classic black, navy, or for a summery vibe, a crisp white or pale grey.
- **Shirt:** Wear a crisp white dress shirt, perhaps with subtle cufflinks for extra elegance.
- **Accessories:** Add a silk pocket square, polished dress shoes, and a sleek watch. If the event allows, a playful patterned bowtie or colored socks can add personality.
- **Grooming:** Keep hair neatly styled and grooming sharp.

**Pro Tip:** If the gala is outdoors, take sunglasses and consider a light shawl (for women) or a linen blazer (for men) for the evening breeze.

Would you like outfit inspiration for a specific color or style?

In [11]:
context = "\n\n".join(data['text'].tolist())
advise  = rag_query("summer gala ball", context)
display(Markdown(advise))

How fabulous! A summer gala ball calls for high-glamour, a touch of drama, and a look that’s both on-trend and event-appropriate. Here are some ultra-stylish outfit ideas, referencing the hottest 2023 fashion trends:

---

### **1. The Showstopper: Red with a Glossy Twist**
- **Dress:** Go for a floor-length, juicy red gown with vibrant orange undertones—think satin, silk, or any glossy finish. Look for sculptural or draped elements for a modern feel.
- **Accessories:** Pair with sculptural silver or gold statement earrings and a matching sculptural clutch.
- **Shoes:** Strappy metallic sandals (silver or gold).
- **Beauty:** Sleek hair and a bold red lip to match the dress for a high-impact effect.

---

### **2. Sheer Sophistication**
- **Dress:** A sheer overlay gown (nude or pastel) with strategic layering or embellishments—think embellished rosettes or 3D florals are ultra on-trend.
- **Accessories:** Delicate drop earrings with a little sparkle; a metallic clutch.
- **Shoes:** Cobalt blue or striking white mules for a fresh pop.
- **Beauty:** Soft, ethereal makeup with a dewy finish and a loose, romantic updo.

---

### **3. Romantic Femininity & Balletcore**
- **Dress:** A pastel-hued tulle or chiffon ball gown—think ballet-inspired, with a full skirt and subtle ruffle or lace details.
- **Accessories:** Dainty pearl necklace or sculptural earrings; satin ballet flats or barely-there metallic heels.
- **Extras:** Optional opera gloves for vintage drama.
- **Beauty:** Think soft waves, flushed cheeks, and glossy lips.

---

### **4. Modern Minimalist with Metallics**
- **Dress:** Sleek, form-fitting dress in liquid silver or gold (think Saint Laurent/Alaïa inspiration), with an interesting neckline or a slit.
- **Accessories:** Oversized statement earrings in gold or silver, a minimal metallic clutch.
- **Shoes:** Barely-there heels, preferably a clear-strap or mirrored finish.
- **Beauty:** Slicked-back ponytail, smoky eyes, and nude lips.

---

### **5. Dreamy Maxi Skirt Moment**
- **Look:** A two-piece set with a silky or organza maxi skirt and a coordinating or contrasting sheer top (with a little sparkle or beaded detail).
- **Accessories:** Sculptural geometric clutch, layered bangles.
- **Shoes:** Shiny platform sandals.
- **Beauty:** Statement eyeliner and glossy highlighter for a radiant finish.

---

#### **Final Touches:**
- **Bag:** Opt for a sculptural or oversized clutch in a metallic or pastel shade.
- **Jewelry:** Choose shoulder-grazing earrings or a 3D floral accessory for a truly fashion-forward moment.
- **Outerwear (if needed):** Light organza or satin shawl.

---

Let me know your color preference or style vibe, and I can tailor the look even more to suit you and ensure you stand out on the dance floor! ✨